***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'Census')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
    path_main = os.path.join(path_sp, 'Data')


path_code    = os.path.join(path_git, 'Data', 'Census')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0,        'Functions.py')).read())
exec(open(os.path.join(path_config , 'Census Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 01a - Prepare API Request Inputs.py')).read())

# View result
df_vars.head(3)

In [ ]:
# If PUMA, change estimate title
if geography == 'PUMA':
    estimate = re.sub('ACS', 'PUMS', estimate)

# Name of the export
end = 'raw.csv'
export_title = f"{indicator_name}_{geography}_{estimate}_{end}"


df_census_raw = pd.read_csv(os.path.join(path_raw, export_title))
df_census_raw.head()

***

Processing

***

In [ ]:
# Execute script to prepare API request inputs
exec(open(os.path.join(path_code, 'Supplemental Scripts', 'Step 02a - Prepare Processing Parameters.py')).read())

In [ ]:
## Make copy of raw data
df_census = df_census_raw.copy()


if sample_type in ['ACS', 'SUBJECT']:

    # Replace weird missing values with np.nan
    # Melt data from wide to long
    # Convert imported values to numeric
    # Manually check column names and clean as needed
    # Adjust dollars for inflation, if needed
    # Reorganize margin of error fields
    df_census = acs_processing_1(df_census, df_vars, geography, margin_of_error)
    display(df_census.head(3))

    # Merge cleam label field, variable mapping, race/ethnicity, and sorting field
    # Remove unneeded columns
    df_census = acs_processing_2(df_census, df_vars, indicator_name, geography, margin_of_error, year_end, path_main, path_git)
    display(df_census.head(3))

    # Create "Categorical" race/ethnicity field for sorting
    # Sort by geography, variable mapping, and race/ethnicity
    # sort and then remove categorical field
    df_census = acs_processing_3(df_census, geography)
    display(df_census.head(3))

    # Final processing step for ACS data
    # Link various FIPS codes
    # Roll up population/households/SE's to the desired geography and variable groupings
    # Calculate percentages by geography, race/ethnicity, and variables
    if geography == 'Places':
        df_places1, df_places2 = acs_processing_4(df_census, indicator_name, geography, percentages, margin_of_error, num_vars, df_fips)
        display(df_places1.head(3), df_places2.head(3))
    if geography == 'Block Groups':
        df_blocks1, df_blocks2 = acs_processing_4(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_blocks1.head(3), df_blocks2.head(3))
    if geography == 'Tracts':
        df_tracts1, df_tracts2 = acs_processing_4(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_tracts1.head(3), df_tracts2.head(3))
    if geography == 'Counties':
        df_counties1, df_counties2, df_mpo1, df_mpo2 = acs_processing_4(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_mpo1.head(3), df_mpo2.head(3))
    if geography == 'MSA':
        df_msa1, df_msa2 = acs_processing_4(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars)
        display(df_msa1.head(3), df_msa2.head(3))



if sample_type in ['PUMS', 'FOODSEC']:
    
    # Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
    # Convert weighted column to integer, convert value fields to string to use as merge field
    # Reshape data dictionary of values/descriptions and reorganize columns
    # Merge meaningful value descriptions onto imported data
    df_census, groups = pums_processing_1(df_census, df_vars, sample_type, weight)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))

    # Remove rows with missing values
    # Only keep description mappings, remove the original PUMS values
    df_census = pums_processing_2(df_census, sample_type, groups, df_fips, dict_fips, path_git)
    display(df_census.head(3))

    # Cleans race/ethnicity fields
    # Creates additional grouping variables for certain indicators
    # Adjusts income variables by inflation for the latest year
    df_census, groups = pums_processing_3(df_census, groups, indicator_name, path_config0)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))


    # Roll up using suggested weight field
    # Roll up to PUMA, counties, MSA, and MPO
    if sample_type == 'PUMS':
        df_puma, df_counties, df_msa, df_mpo, groups = pums_processing_4(df_census, indicator_name, weight, margin_of_error, MOE_thresh, percentages, groups)
        display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))

    # Roll up using suggested weight field
    # Roll up to counties and MPO
    if sample_type == 'FOODSEC':
        df_counties, df_mpo, groups = food_processing_4(df_census, weight, percentages, groups)
        display(df_counties.head(3), df_mpo.head(3))


if estimate == 'LEHD':
    if geography == 'Counties':
        df_counties, df_mpo = lehd_processing(df_census, geography, indicator_name, percentages, df_fips)
        display(df_counties.head(3), df_mpo.head(3))
    if geography == 'MSA':
        df_msa = lehd_processing(df_census, geography, indicator_name, percentages)
        display(df_msa.head(3))
    

In [ ]:

# Final organization of tables for cleanliness
# Renaming columns, subsetting to only desired columns, ...

print('Final Results: ')
print('')

if geography == 'Block Groups':
    df_tracts1 = rename_census(df_blocks1        = df_blocks1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error
                               , percentages     = percentages
                               , sample_type     = sample_type)
    display(df_blocks1.head(3))
if geography == 'Tracts':
    df_tracts1 = rename_census(df_tracts1        = df_tracts1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error
                               , percentages     = percentages
                               , sample_type     = sample_type)
    display(df_tracts1.head(3))
if geography == 'Counties':
    if sample_type in ['ACS', 'SUBJECT']:
        df_counties1, df_mpo1 = rename_census(df_counties1      = df_counties1
                                              , df_mpo1         = df_mpo1
                                              , geography       = geography
                                              , indicator_name  = indicator_name
                                              , margin_of_error = margin_of_error
                                              , percentages     = percentages
                                              , sample_type     = sample_type)
        display(df_counties1.head(3), df_mpo1.head(3))
    if sample_type == 'FOODSEC':
        df_counties, df_mpo = rename_census(df_counties         = df_counties
                                              , df_mpo          = df_mpo
                                              , geography       = geography
                                              , indicator_name  = indicator_name
                                              , margin_of_error = margin_of_error
                                              , sample_type     = sample_type
                                              , table_type      = table_type
                                              , groups          = groups)
        display(df_counties.head(3), df_mpo.head(3))
if geography == 'MSA':
    if sample_type in ['ACS', 'SUBJECT']:
        df_msa1 = rename_census(df_msa1           = df_msa1
                                , geography       = geography
                                , indicator_name  = indicator_name
                                , margin_of_error = margin_of_error
                                , percentages     = percentages
                                , sample_type     = sample_type)
        display(df_msa1.head(3))
if geography == 'PUMA':
    df_puma, df_counties, df_msa, df_mpo = rename_census(df_puma           = df_puma
                                                         , df_counties     = df_counties
                                                         , df_msa          = df_msa
                                                         , df_mpo          = df_mpo
                                                         , geography       = geography
                                                         , indicator_name  = indicator_name
                                                         , margin_of_error = margin_of_error
                                                         , percentages     = percentages
                                                         , sample_type     = sample_type
                                                         , groups          = groups
                                                         , table_type      = table_type)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))


In [ ]:
# MSA mid size national comparison regions and California comparison regions
# SACOG region (MPO)
# Sacramento region MSAs, plus comparator peer regions (at Metropolitan Statistical Area)
# MSAs of both SACOG Peers and SACOG Regions
# SACOG Regions and National Scale
# MSA, MPO, County, Cities, ZIP (SACOG only)
# CA MPOs (region); national mid sized MSAs; County, City ZIP (SACOG only)

In [ ]:
# with pd.ExcelWriter(os.path.join(path_main, 'About Indicators.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
#     df_about.to_excel(writer, index = False, sheet_name = df.columns[1], header = False)


# Create about documentation page for export
if estimate != 'LEHD':
    df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0, MOE_thresh, estimate)
else:
    df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0, estimate)

print("Visual representation of the output for:", indicator_name)
display(df_about)

***

Exporting

***

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, export_loc, f"{indicator_name} {folder}")

if indicator_name == 'EJ_Analysis':
    path_out_xlsx = r'I:\Projects\Warren\Environmental_Justice_March_2023\SACOG_EJ_UPDATE_2024\Python\YOUR_OUTPUT_FOLDER\2 - Processed'

if geography == 'Places':
    workbook_name = f"{indicator_name} Places {estimate}.xlsx"
if geography == 'Block Groups':
    workbook_name = f"{indicator_name} Block Groups {estimate}.xlsx"
if geography == 'Tracts':
    workbook_name = f"{indicator_name} Tracts {estimate}.xlsx"
if geography == 'Counties':
    workbook_name1 = f"{indicator_name} Counties {estimate}.xlsx"
    workbook_name2 = f"{indicator_name} MPO {estimate}.xlsx"
if geography == 'MSA':
    workbook_name = f"{indicator_name} MSA {estimate}.xlsx"

if geography == 'PUMA':
    estimate = re.sub('ACS', 'PUMS', estimate)
    workbook_name1 = f"{indicator_name} PUMA {estimate}.xlsx"
    workbook_name2 = f"{indicator_name} Counties {estimate}.xlsx"
    workbook_name3 = f"{indicator_name} MSA {estimate}.xlsx"
    workbook_name4 = f"{indicator_name} MPO {estimate}.xlsx"
        

In [ ]:
print('Excel files exported here: ' + path_out_xlsx)

if sample_type in ['ACS', 'SUBJECT']:
    if geography == 'Places':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About' , header=False)
            df_places1.to_excel(writer, index = False, sheet_name = 'Places'              )
    if geography == 'Block Groups':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About'       , header=False)
            df_blocks1.to_excel(writer, index = False, sheet_name = 'Block Groups'              )
    if geography == 'Tracts':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About' , header=False)
            df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts'              )
    if geography == 'Counties':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about    .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties'              )
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'MPO'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo1 .to_excel(writer, index = False, sheet_name = 'MPO'                )
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_msa1 .to_excel(writer, index = False, sheet_name = 'MSA'                )


if sample_type == 'PUMS':
    if geography == 'PUMA':
        # # with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
        #     df_about.to_excel(writer, index = False, header=False, sheet_name = 'About')
        #     df_puma .to_excel(writer, index = False, sheet_name = 'PUMA')
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'Counties'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        # df_about.loc[df_about['Metadata'] == 'Geography', 'Description'] = 'MSA'
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name3), engine='xlsxwriter') as writer:
        #     df_about.to_excel(writer, index = False, header=False, sheet_name = 'About')
        #     df_msa  .to_excel(writer, index = False, sheet_name = 'MSA')
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'MPO'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name4),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO')


if sample_type == 'FOODSEC':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO'                )


if estimate == 'LEHD':
    if geography == 'Counties':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO'                )
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_msa  .to_excel(writer, index = False, sheet_name = 'MSA'                )
        

print('')
print("Successfully exported!")

In [ ]:
path_out_xlsx = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"

print('Excel files exported here: ' + path_out_xlsx)

if sample_type in ['ACS', 'SUBJECT']:
    if geography == 'Places':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About' , header=False)
            df_places1.to_excel(writer, index = False, sheet_name = 'Places'              )
    if geography == 'Block Groups':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About'       , header=False)
            df_blocks1.to_excel(writer, index = False, sheet_name = 'Block Groups'              )
    if geography == 'Tracts':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about  .to_excel(writer, index = False, sheet_name = 'About' , header=False)
            df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts'              )
    if geography == 'Counties':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about    .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties1.to_excel(writer, index = False, sheet_name = 'Counties'              )
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'MPO'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo1 .to_excel(writer, index = False, sheet_name = 'MPO'                )
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_msa1 .to_excel(writer, index = False, sheet_name = 'MSA'                )


if sample_type == 'PUMS':
    if geography == 'PUMA':
        # # with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
        #     df_about.to_excel(writer, index = False, header=False, sheet_name = 'About')
        #     df_puma .to_excel(writer, index = False, sheet_name = 'PUMA')
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'Counties'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        # df_about.loc[df_about['Metadata'] == 'Geography', 'Description'] = 'MSA'
        # with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name3), engine='xlsxwriter') as writer:
        #     df_about.to_excel(writer, index = False, header=False, sheet_name = 'About')
        #     df_msa  .to_excel(writer, index = False, sheet_name = 'MSA')
        df_about.loc[df_about['Indicator'] == 'Geography', 'Description'] = 'MPO'
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name4),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO')


if sample_type == 'FOODSEC':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO'                )


if estimate == 'LEHD':
    if geography == 'Counties':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name1), engine='xlsxwriter') as writer:
            df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header=False)
            df_counties.to_excel(writer, index = False, sheet_name = 'Counties'              )
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name2), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_mpo  .to_excel(writer, index = False, sheet_name = 'MPO'                )
    if geography == 'MSA':
        with pd.ExcelWriter(os.path.join(path_out_xlsx, workbook_name), engine='xlsxwriter') as writer:
            df_about.to_excel(writer, index = False, sheet_name = 'About', header=False)
            df_msa  .to_excel(writer, index = False, sheet_name = 'MSA'                )
        

print('')
print("Successfully exported!")